# Adaptive Hybrid RecSys — Training (self-contained)

**Требование:** запущен `01_data_pipeline.ipynb`, данные лежат в `MyDrive/disser/data/processed/`

**Порядок:** Runtime → Change runtime type → **T4 GPU** → Run all

In [ ]:
# ── 1. Mount Drive + paths ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR     = '/content/drive/MyDrive/disser'
PROCESSED_DIR = f'{DRIVE_DIR}/data/processed'
RECBOLE_DIR   = f'{DRIVE_DIR}/data/recbole'
OUTPUT_DIR    = f'{DRIVE_DIR}/outputs'

for d in [OUTPUT_DIR, f'{OUTPUT_DIR}/checkpoints', f'{OUTPUT_DIR}/figures']:
    os.makedirs(d, exist_ok=True)

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# ── 2. Install deps ────────────────────────────────────────────────────────
!pip install -q recbole optuna mlflow pyarrow
print('Done')

In [ ]:
# ── 3. Load processed data from Drive ─────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.sparse as sp
import json, gc
from pathlib import Path

P = Path(PROCESSED_DIR)
stats = json.load(open(P / 'dataset_stats.json'))
print('Dataset stats:', json.dumps(stats, indent=2))

print('\nLoading splits...')
train_df       = pd.read_parquet(P / 'train.parquet')
val_df         = pd.read_parquet(P / 'val.parquet')
test_df        = pd.read_parquet(P / 'test.parquet')
train_temporal = pd.read_parquet(P / 'train_temporal.parquet')
val_temporal   = pd.read_parquet(P / 'val_temporal.parquet')
test_temporal  = pd.read_parquet(P / 'test_temporal.parquet')

print('Loading TF-IDF (sparse)...')
item_content = sp.load_npz(str(P / 'item_content_sparse.npz'))

print('Loading user sequences...')
seq_data       = np.load(str(P / 'user_sequences.npz'), allow_pickle=True)
user_sequences = dict(seq_data['sequences'].item())

# train_user_items for negative sampling / eval
train_user_items = {}
for uid, iid in zip(train_df['user_idx'].values, train_df['item_idx'].values):
    train_user_items.setdefault(int(uid), set()).add(int(iid))

N_USERS     = stats['n_users']
N_ITEMS     = stats['n_items']
CONTENT_DIM = stats['content_dim']
CONTEXT_DIM = stats['context_dim']

print(f'\nUsers: {N_USERS:,} | Items: {N_ITEMS:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'TF-IDF: {item_content.shape}, nnz={item_content.nnz:,}')

In [ ]:
# ── 4. Dataset & DataLoader ────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 50
BATCH_SIZE  = 1024

def get_content_row(mat, idx, dim):
    if idx >= mat.shape[0]:
        return np.zeros(dim, dtype=np.float32)
    row = mat[idx]
    return row.toarray().squeeze().astype(np.float32) if hasattr(row, 'toarray') else np.asarray(row, dtype=np.float32)

class RecDataset(Dataset):
    def __init__(self, df, temporal_df, neg_sample=True):
        self.df       = df.reset_index(drop=True)
        self.temporal = temporal_df.values.astype(np.float32)
        self.neg      = neg_sample
        self.user_items = {}
        for uid, iid in zip(self.df['user_idx'].values, self.df['item_idx'].values):
            self.user_items.setdefault(int(uid), set()).add(int(iid))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        user_idx    = int(row['user_idx'])
        pos_item    = int(row['item_idx'])

        if self.neg:
            user_set = self.user_items.get(user_idx, set())
            neg_item = np.random.randint(1, N_ITEMS + 1)
            while neg_item in user_set:
                neg_item = np.random.randint(1, N_ITEMS + 1)
        else:
            neg_item = 0

        seq     = user_sequences.get(user_idx, np.array([], dtype=np.int64))
        seq_len = min(len(seq), MAX_SEQ_LEN)
        padded  = np.zeros(MAX_SEQ_LEN, dtype=np.int64)
        if seq_len > 0: padded[-seq_len:] = seq[-seq_len:]

        return {
            'user_idx':     torch.tensor(user_idx,  dtype=torch.long),
            'pos_item_idx': torch.tensor(pos_item,  dtype=torch.long),
            'neg_item_idx': torch.tensor(neg_item,  dtype=torch.long),
            'sequence':     torch.tensor(padded,    dtype=torch.long),
            'seq_len':      torch.tensor(max(seq_len, 1), dtype=torch.long),
            'pos_content':  torch.tensor(get_content_row(item_content, pos_item, CONTENT_DIM), dtype=torch.float32),
            'neg_content':  torch.tensor(get_content_row(item_content, neg_item, CONTENT_DIM), dtype=torch.float32),
            'context':      torch.tensor(self.temporal[idx] if idx < len(self.temporal) else np.zeros(CONTEXT_DIM, np.float32), dtype=torch.float32),
        }

train_loader = DataLoader(RecDataset(train_df, train_temporal, neg_sample=True),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(RecDataset(val_df,   val_temporal,   neg_sample=True),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(RecDataset(test_df,  test_temporal,  neg_sample=True),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'DataLoaders ready | train={len(train_loader.dataset):,} val={len(val_loader.dataset):,} test={len(test_loader.dataset):,}')

In [ ]:
# ── 5. Model definition ────────────────────────────────────────────────────
import torch.nn as nn

EMBED_DIM = 64

class StaticComponent(nn.Module):
    """NCF: user + item embeddings → MLP."""
    def __init__(self):
        super().__init__()
        self.user_emb = nn.Embedding(N_USERS + 1, EMBED_DIM, padding_idx=0)
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(EMBED_DIM * 2, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64),            nn.ReLU(),
            nn.Linear(64, EMBED_DIM),
        )
    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)
        i = self.item_emb(item_ids)
        return self.mlp(torch.cat([u, i], dim=-1))

class DynamicComponent(nn.Module):
    """GRU over item sequence."""
    def __init__(self):
        super().__init__()
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.gru = nn.GRU(EMBED_DIM, 128, num_layers=2, batch_first=True, dropout=0.1)
        self.proj = nn.Linear(128, EMBED_DIM)
    def forward(self, seqs, lengths):
        x = self.item_emb(seqs)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.proj(h[-1])

class ContentComponent(nn.Module):
    """MLP over TF-IDF features."""
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(CONTENT_DIM, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),         nn.ReLU(),
            nn.Linear(256, EMBED_DIM),
        )
    def forward(self, x): return self.mlp(x)

class AttentionGate(nn.Module):
    """Context → softmax weights for 3 components."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(CONTEXT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 3),
        )
    def forward(self, ctx):
        return torch.softmax(self.net(ctx), dim=-1)

class AdaptiveHybridModel(nn.Module):
    def __init__(self, use_dynamic=True, use_content=True, use_attention=True):
        super().__init__()
        self.use_dynamic   = use_dynamic
        self.use_content   = use_content
        self.use_attention = use_attention
        self.static  = StaticComponent()
        self.dynamic = DynamicComponent()
        self.content = ContentComponent()
        self.gate    = AttentionGate()
        self.head    = nn.Sequential(
            nn.Linear(EMBED_DIM, EMBED_DIM // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(EMBED_DIM // 2, 1),
        )

    def forward(self, user_ids, item_ids, seqs, lengths, content, ctx):
        h_s = self.static(user_ids, item_ids)
        h_d = self.dynamic(seqs, lengths)  if self.use_dynamic  else torch.zeros_like(h_s)
        h_c = self.content(content)        if self.use_content  else torch.zeros_like(h_s)

        stack   = torch.stack([h_s, h_d, h_c], dim=1)  # (B, 3, E)
        weights = self.gate(ctx).unsqueeze(-1) if self.use_attention else torch.ones(stack.size(0), 3, 1, device=stack.device) / 3
        fused   = (stack * weights).sum(dim=1)
        return self.head(fused).squeeze(-1)

model = AdaptiveHybridModel().to(DEVICE)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 6. Training helpers ────────────────────────────────────────────────────
import time
from tqdm.notebook import tqdm

EPOCHS    = 100   # потолок; early stopping остановит раньше (~20-40 эпох)
LR        = 0.001
PATIENCE  = 7     # чуть больше терпения для крупного датасета
CKPT_PATH = f'{OUTPUT_DIR}/checkpoints/best_model.pt'

def bpr_loss(pos, neg):
    return -torch.log(torch.sigmoid(pos - neg) + 1e-8).mean()

def evaluate(m, loader, n_sample=100, k=10):
    """Evaluate model m on loader. Returns {Recall@10, NDCG@10}."""
    m.eval()
    recalls, ndcgs = [], []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            u   = batch['user_idx'].to(DEVICE)
            pi  = batch['pos_item_idx'].to(DEVICE)
            seq = batch['sequence'].to(DEVICE)
            sl  = batch['seq_len'].to(DEVICE)
            pc  = batch['pos_content'].to(DEVICE)
            ctx = batch['context'].to(DEVICE)

            for i in range(u.size(0)):
                pos_item = pi[i].item()
                uid      = u[i].item()
                train_s  = train_user_items.get(uid, set())

                negs = []
                while len(negs) < n_sample:
                    c = np.random.randint(1, N_ITEMS + 1)
                    if c not in train_s and c != pos_item:
                        negs.append(c)

                all_items = [pos_item] + negs
                it  = torch.tensor(all_items, dtype=torch.long, device=DEVICE)
                uu  = u[i].unsqueeze(0).expand(len(all_items))
                ss  = seq[i].unsqueeze(0).expand(len(all_items), -1)
                sll = sl[i].unsqueeze(0).expand(len(all_items))
                cc  = ctx[i].unsqueeze(0).expand(len(all_items), -1)
                cnt = pc[i].unsqueeze(0).expand(len(all_items), -1)

                scores  = m(uu, it, ss, sll, cnt, cc)
                _, idxs = torch.sort(scores, descending=True)
                ranked  = [all_items[j] for j in idxs.cpu().numpy()]

                hits = [1 if ranked[j] == pos_item else 0 for j in range(k)]
                recalls.append(1 if pos_item in ranked[:k] else 0)
                dcg  = sum(h / np.log2(j+2) for j, h in enumerate(hits))
                ndcgs.append(dcg)

    return {'Recall@10': np.mean(recalls), 'NDCG@10': np.mean(ndcgs)}

def train_epoch(m, opt):
    m.train()
    total, n = 0.0, 0
    for batch in tqdm(train_loader, leave=False):
        u   = batch['user_idx'].to(DEVICE)
        pi  = batch['pos_item_idx'].to(DEVICE)
        ni  = batch['neg_item_idx'].to(DEVICE)
        seq = batch['sequence'].to(DEVICE)
        sl  = batch['seq_len'].to(DEVICE)
        pc  = batch['pos_content'].to(DEVICE)
        nc  = batch['neg_content'].to(DEVICE)
        ctx = batch['context'].to(DEVICE)

        loss = bpr_loss(m(u, pi, seq, sl, pc, ctx), m(u, ni, seq, sl, nc, ctx))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        total += loss.item(); n += 1
    return total / max(n, 1)

# ── Main model training ────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_ndcg, patience_cnt, history = 0.0, 0, []

for epoch in range(EPOCHS):
    t0   = time.time()
    loss = train_epoch(model, optimizer)
    scheduler.step()
    val_m = evaluate(model, val_loader)
    elapsed = time.time() - t0

    history.append({'epoch': epoch+1, 'loss': loss, **val_m})
    print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss {loss:.4f} | NDCG@10 {val_m["NDCG@10"]:.4f} | Recall@10 {val_m["Recall@10"]:.4f} | {elapsed:.0f}s')

    if val_m['NDCG@10'] > best_ndcg:
        best_ndcg = val_m['NDCG@10']
        patience_cnt = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch+1, 'val_metrics': val_m}, CKPT_PATH)
        print(f'  ✓ Checkpoint saved (best NDCG@10={best_ndcg:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
            break

print(f'\nTraining done. Best NDCG@10={best_ndcg:.4f}')

In [ ]:
# ── 7. Test set evaluation ─────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val NDCG@10={ckpt['val_metrics']['NDCG@10']:.4f})")

test_metrics = evaluate(model, test_loader, n_sample=100, k=10)
print(f'\nTest metrics: {test_metrics}')

import json
results = {
    'model': 'AdaptiveHybridModel',
    'test_metrics': test_metrics,
    'best_val_ndcg': best_ndcg,
    'n_users': N_USERS, 'n_items': N_ITEMS,
    'total_params': sum(p.numel() for p in model.parameters()),
    'history': history,
}
with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved to {OUTPUT_DIR}/results.json')

In [ ]:
# ── 8. Ablation study ─────────────────────────────────────────────────────
ABLATION_VARIANTS = {
    'full_model':   (True,  True,  True),
    'no_dynamic':   (False, True,  True),
    'no_content':   (True,  False, True),
    'no_attention': (True,  True,  False),
    'static_only':  (False, False, False),
}

ablation_results = {}
ABLATION_EPOCHS  = 20

for variant, (use_dyn, use_cnt, use_att) in ABLATION_VARIANTS.items():
    print(f'\n=== {variant} ===')
    m   = AdaptiveHybridModel(use_dynamic=use_dyn, use_content=use_cnt, use_attention=use_att).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=1e-4)
    best_v, pat, best_state = 0.0, 0, None

    for ep in range(ABLATION_EPOCHS):
        loss = train_epoch(m, opt)
        val_m = evaluate(m, val_loader, n_sample=50, k=10)  # faster: 50 negatives
        print(f'  ep {ep+1}/{ABLATION_EPOCHS} | loss={loss:.4f} | NDCG@10={val_m["NDCG@10"]:.4f}')

        if val_m['NDCG@10'] > best_v:
            best_v = val_m['NDCG@10']; pat = 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            pat += 1
            if pat >= 3:
                print('  early stop'); break

    if best_state:
        m.load_state_dict(best_state)
    test_m = evaluate(m, test_loader, n_sample=100, k=10)
    ablation_results[variant] = test_m
    print(f'  → Test: {test_m}')

with open(f'{OUTPUT_DIR}/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

print('\n' + '='*55)
print(f'{"Variant":<20} {"Recall@10":>12} {"NDCG@10":>12}')
print('-'*45)
for name, m2 in ablation_results.items():
    print(f"{name:<20} {m2.get('Recall@10',0):>12.4f} {m2.get('NDCG@10',0):>12.4f}")
print('='*55)

In [ ]:
# ── 9. Baseline models (BPR + Pop через RecBole) ───────────────────────────
from recbole.quick_start import run_recbole
import os, json

# Конфиги RecBole (инлайн)
BPR_CFG = {
    'data_path': RECBOLE_DIR,
    'embedding_size': 64,
    'epochs': 50, 'train_batch_size': 2048, 'learning_rate': 0.001,
    'eval_args': {'group_by': 'user', 'order': 'TO', 'split': {'RS': [0.7, 0.15, 0.15]}, 'mode': 'full'},
    'metrics': ['Recall', 'NDCG', 'Precision'], 'topk': [10], 'valid_metric': 'NDCG@10',
    'USER_ID_FIELD': 'user_id', 'ITEM_ID_FIELD': 'item_id', 'TIME_FIELD': 'timestamp',
    'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
    'device': str(DEVICE),
}
POP_CFG = {**BPR_CFG, 'epochs': 1}

baseline_results = {}
for model_name, cfg in [('Pop', POP_CFG), ('BPR', BPR_CFG)]:
    print(f'\nRunning {model_name}...')
    result = run_recbole(model=model_name, dataset='amazon-electronics', config_dict=cfg)
    tr = result.get('test_result', {})
    baseline_results[model_name] = {str(k): float(v) for k, v in tr.items()}
    print(f'{model_name}: {baseline_results[model_name]}')

with open(f'{OUTPUT_DIR}/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('Baselines saved.')

In [ ]:
# ── 10. Final comparison table ─────────────────────────────────────────────
import json

results   = json.load(open(f'{OUTPUT_DIR}/results.json'))
baselines = json.load(open(f'{OUTPUT_DIR}/baseline_results.json'))
ablation  = json.load(open(f'{OUTPUT_DIR}/ablation_results.json'))

print('='*65)
print('FINAL COMPARISON')
print('='*65)
print(f'{"Model":<25} {"Recall@10":>12} {"NDCG@10":>12}')
print('-'*50)
for name, m2 in baselines.items():
    r = m2.get('recall@10', m2.get('Recall@10', 0))
    n = m2.get('ndcg@10',   m2.get('NDCG@10', 0))
    print(f'{name:<25} {r:>12.4f} {n:>12.4f}')
print('-'*50)
for name, m2 in ablation.items():
    print(f'{name:<25} {m2.get("Recall@10",0):>12.4f} {m2.get("NDCG@10",0):>12.4f}')
print('-'*50)
m2 = results['test_metrics']
print(f'{"AdaptiveHybrid (ours)":<25} {m2.get("Recall@10",0):>12.4f} {m2.get("NDCG@10",0):>12.4f}  ← best?')
print('='*65)
print(f'All outputs: {OUTPUT_DIR}')